# Semana 7, días 3 y 4

Afinar un modelo de código abierto para predecir los precios de los productos

Consulte este cuaderno en Google Colab:

https://colab.research.google.com/drive/1_tpgQdLt1ef9LCGZFb542rMFY_UnI-bn?usp=sharing

# Predecir precios de productos

## Día 3: Entrenamiento!


In [ ]:
# pip installs

!pip install -q datasets==2.21.0 requests torch peft bitsandbytes transformers trl accelerate sentencepiece wandb matplotlib

In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
# Constantes

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
PROJECT_NAME = "pricer"
HF_USER = "joanby" # tu nombre de HF va aquí!

# Data

DATASET_NAME = f"{HF_USER}/pricer-data"
# O simplemente usa el que he subido
# DATASET_NAME = "joanby/pricer-data"
MAX_SEQUENCE_LENGTH = 182

# Nombre de la ejecución para guardar el modelo en el Hub

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hiper parámetros de QLoRA

LORA_R = 32 #8
LORA_ALPHA = 64 #16
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1
QUANT_4_BIT = True

# Hiper parámetros para el Entrenamiento

EPOCHS = 3
BATCH_SIZE = 16 #2, 4, 16
GRADIENT_ACCUMULATION_STEPS = 1 #2, 4
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = 'cosine'
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"

# Configuración de Admin

STEPS = 50
SAVE_STEPS = 5000
LOG_TO_WANDB = True

%matplotlib inline

In [ ]:
HUB_MODEL_NAME

# Más sobre optimizadores

https://huggingface.co/docs/transformers/main/en/perf_train_gpu_one#optimizer-choice

El más común es Adam o AdamW (Adam con decaimiento de peso).
Adam logra una buena convergencia al almacenar el promedio móvil de los gradientes anteriores; sin embargo, agrega una huella de memoria adicional del orden de la cantidad de parámetros del modelo.

### Inicia sesión en HuggingFace y Weights & Biases

Si aún no tienes una cuenta de HuggingFace, visita https://huggingface.co para registrarte y crear un token.

Luego, selecciona los secretos para este cuaderno haciendo clic en el ícono de la llave a la izquierda y agrega un nuevo secreto llamado `HF_TOKEN` con el valor como tu token.

Repite esto para weightsandbiases en https://wandb.ai y agrega un secreto llamado `WANDB_API_KEY`

In [ ]:
# Log in en HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Log in en Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configuramos Weights & Biases para almacenar la info de nuestro proyecto
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
test = dataset['test']

In [ ]:
test[0]

In [ ]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

## Ahora cargue el tokenizador y el modelo

El modelo está "cuantificado": estamos reduciendo la precisión a 4 bits.

In [ ]:
# Elegimos la opción de cuantización adecuada

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

In [ ]:
# Cargamos el Tokenizer y el Modelo

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

# Recopilador de datos

Es importante que nos aseguremos durante el entrenamiento de que no estamos intentando entrenar al modelo para predecir la descripción de los productos, sino solo su precio.

Debemos decirle al entrenador que todo hasta "El precio es $" está ahí para dar contexto al modelo para predecir el siguiente token, pero no es necesario aprenderlo.

El entrenador debe enseñarle al modelo a predecir el token o tokens después de "El precio es $".

Hay una forma complicada de hacer esto configurando máscaras, pero afortunadamente HuggingFace proporciona una clase auxiliar súper simple que se encarga de esto por nosotros.

In [ ]:
# `DataCollatorForCompletionOnlyLM` ya no existe en trl. Servía para que la pérdida se calculase
# SOLO sobre el precio y no sobre todo el prompt, que es justo lo que queremos entrenar.
# El equivalente actual es `completion_only_loss=True` en SFTConfig, pero exige que el dataset
# venga en formato prompt/completion en vez de una sola columna de texto, así que partimos el
# texto por el mismo "Price is $" que usaba el collator. El efecto sobre el entrenamiento es el
# mismo: el modelo solo se penaliza por el precio que predice.
response_template = "Price is $"

def to_prompt_completion(example):
    prompt, _, completion = example["text"].partition(response_template)
    return {"prompt": prompt + response_template, "completion": completion}

# num_proc=1 a propósito: con más procesos esto deja workers colgados en algunos entornos.
train = train.map(to_prompt_completion, remove_columns=train.column_names, num_proc=1)
print(train[0]["prompt"][-80:], "||", train[0]["completion"])

# Y AHORA

## Configuramos la configuración para el entrenamiento

Necesitamos crear 2 objetos:

Un objeto LoraConfig con nuestros hiperparámetros para LoRA

Un SFTConfig con nuestros parámetros generales de entrenamiento

In [ ]:
# `warmup_ratio` ya no existe en SFTConfig/TrainingArguments (ni en trl ni en transformers);
# se traduce a su equivalente exacto en número de pasos, conservando el mismo 3% de WARMUP_RATIO.
steps_per_epoch = math.ceil(len(train) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))
WARMUP_STEPS = int(steps_per_epoch * EPOCHS * WARMUP_RATIO)
print(f"warmup_steps = {WARMUP_STEPS} (equivale al {WARMUP_RATIO:.0%} de {steps_per_epoch*EPOCHS} pasos)")

# Primero, especifique los parámetros de configuración para LoRA

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

#A continuación, especifique los parámetros de configuración general para el entrenamiento.

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    # `report_to=None` ya no se acepta: hay que pasar la cadena "none" para desactivar el reporting
    report_to="wandb" if LOG_TO_WANDB else "none",
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    completion_only_loss=True,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True
)

# Y ahora, el entrenador de ajuste fino supervisado realizará el ajuste fino
# Dados estos 2 conjuntos de parámetros de configuración

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora_parameters,
    processing_class=tokenizer,
    args=train_parameters,
)

In [ ]:
# Fine-tune!
fine_tuning.train()

# Subimos nuestro modelo ajustado a Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Almacenado en el hub: {PROJECT_RUN_NAME}")

In [ ]:
if LOG_TO_WANDB:
  wandb.finish()